# Phase 3 — Corpus expansion + promptfix re-audit + leak-free CV

**Session date**: 2026-05-17 / 2026-05-18
**Branch**: `phase2-recomputes-on-v2-gold`
**Goal**: устранить системные источники шума в hybrid cascade и получить честные cross-validated цифры.

## Ход работы
1. v2 vs v3 honest 80/20 holdout (предыдущая сессия) — v3 +2.57pp acc, -11pp cov
2. **v3b (tier-weighted)** silver=1, t1=6, t2=4, t3=2 — best на single holdout (89.13%) и на OFF-truth (78.04%)
3. v3c (silver=0) — slightly worse, silver НЕ просто шум
4. **OFF-derived truth** ×27 expansion eval (40k rows для derived attrs)
5. **Silver-leak fix**: trainer передавал silver labels holdout-кодов → inflate ~0.57pp
6. **5-fold leak-free CV** на Tier1+2: 88.45% ± 0.57% (95% CI ±0.50pp)
7. **Promptfix re-audit**:
   - Прежний промпт не содержал nutriments → Opus/gemini заполняли nutri_score/protein_class в 4-7% случаев
   - Pilot подтвердил: с фиксом — 90-97% fill
   - Full re-audit: Opus 5,930 codes ($60+12=72), gemini 34,433 codes ($15)
8. **v3d retrain** на чистом корпусе (Tier 1 promptfix + Tier 3 promptfix, no gpt-5.5)

## Cost summary
| Item | Cost |
|---|---|
| Phase 1 historical | ~$200 (квота исчерпана) |
| Opus retry v2 promptfix | ~$11 |
| Opus expand 4.5 (5283 codes) | ~$60 |
| Opus expand retry | ~$12 |
| Gemini retry promptfix | ~$15 |
| GPT-5.5 retry (killed early) | ~$4 |
| **Total session** | **~$102** |


In [1]:
import warnings; warnings.filterwarnings('ignore')
import json, glob
import pandas as pd
import numpy as np
from pathlib import Path
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

## 1. Lineage labels

In [2]:
# Tier hierarchy after all annotation rounds
print('SILVER (OFF tags-derived, no LLM):')
for cat in ['pasta','chocolate','cheeses']:
    s = pd.read_parquet(f'../datasets/processed/{cat}_stratified_silver_standard.parquet')
    print(f'  {cat}: {len(s):,} rows')

print('\nTIER 1 — Opus blind OFF-grounded:')
print('  Phase 1 (old prompt): 717 codes (deprecated)')
print('  Phase 1 PROMPTFIX (new prompt): 717 + retry = ~743 codes')
print('  Expand 4.5 PROMPTFIX: 5,313 codes (4359+954 retry)')
print('  Total Tier 1 (promptfix): 6,056 codes (3,058× expansion vs original 239/cat)')

print('\nTIER 2 — gpt-5.5 expansion (deprecated for v3d):')
print('  1,949 codes (old prompt, derived attrs mostly null)')
print('  gpt-5.5 promptfix retry KILLED — too slow/expensive (~$45 + 2h)')

print('\nTIER 3 — gemini-2.5-flash B3 (38k codes):')
print('  Old prompt: 38,747 codes (used in v3/v3b/v3c)')
print('  Promptfix: 34,433/38,947 codes (88.4% coverage)')

print('\nDIRECT LLM (production validation, partner_input — без OFF):')
print('  sonnet-4.5, gpt-4o, gemini-2.5-flash, gpt-oss-120b pilots (~200-300 codes/cat)')
print('  Used as semi-independent ground truth, NOT in training')

SILVER (OFF tags-derived, no LLM):
  pasta: 15,513 rows
  chocolate: 13,269 rows
  cheeses: 21,011 rows

TIER 1 — Opus blind OFF-grounded:
  Phase 1 (old prompt): 717 codes (deprecated)
  Phase 1 PROMPTFIX (new prompt): 717 + retry = ~743 codes
  Expand 4.5 PROMPTFIX: 5,313 codes (4359+954 retry)
  Total Tier 1 (promptfix): 6,056 codes (3,058× expansion vs original 239/cat)

TIER 2 — gpt-5.5 expansion (deprecated for v3d):
  1,949 codes (old prompt, derived attrs mostly null)
  gpt-5.5 promptfix retry KILLED — too slow/expensive (~$45 + 2h)

TIER 3 — gemini-2.5-flash B3 (38k codes):
  Old prompt: 38,747 codes (used in v3/v3b/v3c)
  Promptfix: 34,433/38,947 codes (88.4% coverage)

DIRECT LLM (production validation, partner_input — без OFF):
  sonnet-4.5, gpt-4o, gemini-2.5-flash, gpt-oss-120b pilots (~200-300 codes/cat)
  Used as semi-independent ground truth, NOT in training


## 2. v2 vs v3 vs v3b vs v3c — single 80/20 holdout (Tier1+2, OLD silver-leak)

In [3]:
cmp = pd.read_parquet('../datasets/processed/holdout_eval_4way.parquet')
overall = pd.DataFrame([{
    'acc_v2': cmp['acc_v2'].mean(), 'cov_v2': cmp['cov_v2'].mean(),
    'acc_v3': cmp['acc_v3'].mean(), 'cov_v3': cmp['cov_v3'].mean(),
    'acc_v3b': cmp['acc_v3b'].mean(), 'cov_v3b': cmp['cov_v3b'].mean(),
    'acc_v3c': cmp['acc_v3c'].mean(), 'cov_v3c': cmp['cov_v3c'].mean(),
}])
print('OVERALL MEAN (NOTE: contains silver-leak ~0.5pp inflate):')
print(overall.to_string(index=False, float_format='%.4f'))
print('\nWinner: v3b on accuracy (+2.94pp over v2)')

OVERALL MEAN (NOTE: contains silver-leak ~0.5pp inflate):
 acc_v2  cov_v2  acc_v3  cov_v3  acc_v3b  cov_v3b  acc_v3c  cov_v3c
 0.8619  0.9757  0.8876  0.8633   0.8913   0.8734   0.8847   0.8654

Winner: v3b on accuracy (+2.94pp over v2)


## 3. OFF-derived truth holdout (×27 expansion)

Для derived attrs (nutri_score_grade, protein_class, fat_class) истина = детерминированная формула из OFF nutriments. 56,756 truth-строк vs 1,600 в Tier1+2 holdout.

In [4]:
truth = pd.read_parquet('../datasets/processed/off_derived_truth.parquet')
print(f'Total OFF-derived truth rows: {len(truth):,}')
print('Per category × attr:')
print(truth.groupby(['category','attr']).size().unstack(fill_value=0))

Total OFF-derived truth rows: 56,756
Per category × attr:
attr       fat_class  nutri_score_grade  protein_class
category                                              
cheeses         5767              18272           5766
chocolate          0              10767           1385
pasta              0              11267           3532


In [5]:
off_eval = pd.read_parquet('../datasets/processed/eval_off_truth_4way.parquet')
print('Per-attr (acc on 4500+ OFF-derived truth obs):')
cols = ['category','attr','n_total','n_v2','acc_v2','cov_v2','acc_v3b','cov_v3b','acc_v3c','cov_v3c']
print(off_eval[cols].to_string(index=False, float_format='%.3f'))

Per-attr (acc on 4500+ OFF-derived truth obs):
 category              attr  n_total  n_v2  acc_v2  cov_v2  acc_v3b  cov_v3b  acc_v3c  cov_v3c
  cheeses         fat_class     2808  1880   0.568   0.670    0.815    0.964    0.812    0.968
chocolate nutri_score_grade      465   447   0.828   0.961    0.848    0.880    0.840    0.916
chocolate     protein_class      164   164   0.756   1.000    0.656    0.994    0.648    0.988
    pasta nutri_score_grade      623   501   0.758   0.804    0.753    0.806    0.714    0.886
    pasta     protein_class      390   386   0.839   0.990    0.830    0.979    0.822    0.982


In [6]:
# Overall mean on OFF-truth (4500+ obs, robust CI)
print('OVERALL MEAN (OFF-truth holdout, post-leak-fix):')
print(f'  v2:  acc={off_eval["acc_v2"].mean():.4f}  cov={off_eval["cov_v2"].mean():.4f}')
print(f'  v3:  acc={off_eval["acc_v3"].mean():.4f}  cov={off_eval["cov_v3"].mean():.4f}')
print(f'  v3b: acc={off_eval["acc_v3b"].mean():.4f}  cov={off_eval["cov_v3b"].mean():.4f}')
print(f'  v3c: acc={off_eval["acc_v3c"].mean():.4f}  cov={off_eval["cov_v3c"].mean():.4f}')
print('\nKey finding: cheeses/fat_class v2->v3+ = +24.7pp (gemini unlocked)')
print('Regression: chocolate/protein_class -13pp (gemini old-prompt nulled — v3d should fix)')

OVERALL MEAN (OFF-truth holdout, post-leak-fix):
  v2:  acc=0.7500  cov=0.8849
  v3:  acc=0.7686  cov=0.9330
  v3b: acc=0.7804  cov=0.9246
  v3c: acc=0.7673  cov=0.9479

Key finding: cheeses/fat_class v2->v3+ = +24.7pp (gemini unlocked)
Regression: chocolate/protein_class -13pp (gemini old-prompt nulled — v3d should fix)


## 4. Silver-leak finding + fix

**Bug**: trainer передавал `silver_keep` без фильтра по holdout-кодам. Для каждого holdout-кода silver label (`~85-90% acc`) ВКЛЮЧАЛСЯ в обучение, потом тот же код тестировался → утечка.

**Fix**: добавлен `--holdout-codes` arg в `train_hybrid_cascade.py`. Silver и gold отфильтровываются.

**Magnitude**: leak boosted accuracy by ~0.57pp (fold 0 leaky=89.22% → leak-free=88.65%).

n_silver упал на 17-32% за счёт исключения holdout-кодов:
- pasta/grain_type: 334 → 226 (-108)
- pasta/is_filled: 539 → 378 (-161)
- pasta/nutri_score_grade: 786 → 654 (-132)


## 5. 5-fold CV leak-free (v3b config)

In [7]:
cv = pd.read_parquet('../datasets/processed/cv5fold_v3b.parquet')
print('Per-fold:')
print(cv.to_string(index=False, float_format='%.4f'))
print(f'\nAccuracy: {cv["acc"].mean():.4f} ± {cv["acc"].std(ddof=1):.4f}')
print(f'Coverage: {cv["cov"].mean():.4f} ± {cv["cov"].std(ddof=1):.4f}')
# Bootstrap CI
np.random.seed(42)
bs = sorted(np.random.choice(cv['acc'].values, size=(1000, len(cv)), replace=True).mean(axis=1))
print(f'Bootstrap 95% CI: [{bs[25]:.4f}, {bs[975]:.4f}]')

Per-fold:
 fold    acc    cov  n_train  n_holdout
    0 0.8865 0.8623     2132       3596
    1 0.8873 0.8622     2132       3571
    2 0.8912 0.8607     2132       3619
    3 0.8769 0.8605     2133       3607
    4 0.8806 0.8659     2135       3580

Accuracy: 0.8845 ± 0.0057
Coverage: 0.8624 ± 0.0022
Bootstrap 95% CI: [0.8797, 0.8887]


## 6. Promptfix re-audit — pilot validation

Старый промпт не содержал `nutriments` block → derived attrs (nutri_score, protein, fat) почти всегда null.
Pilot на 30 codes/cat показал драматический эффект:

| attr | Phase 1 fill | Promptfix fill | Δ |
|---|---|---|---|
| pasta/nutri_score_grade | 4% | 97% | +93pp |
| pasta/protein_class | 4% | 97% | +93pp |
| chocolate/nutri_score_grade | 7% | 90% | +83pp |
| chocolate/protein_class | 7% | 90% | +83pp |
| cheeses/fat_class | 97% | 100% | +3pp |

→ Full re-audit запущен: Opus 5,930 codes (~$72), gemini 34,433 codes (~$15).


## 7. v3d build (consensus_v3d.parquet)

In [8]:
v3d = pd.read_parquet('../datasets/processed/consensus_v3d.parquet')
print(f'v3d: {len(v3d):,} rows, {v3d["code"].nunique():,} codes')
print('\nPer category × tier (codes):')
print(v3d.groupby(['category','tier'])['code'].nunique().unstack(fill_value=0))

v3d: 259,299 rows, 35,707 codes

Per category × tier (codes):
tier       tier1_opus_expand45  tier1_opus_phase1  tier3_gemini_promptfix
category                                                                 
cheeses                   1762                246                   10145
chocolate                 1770                247                    9688
pasta                     1781                250                    9818


## 8. v3d eval (TODO — fill after retrain + eval)

In [9]:
# After v3d retrain finishes, this loads new comparison
try:
    v3d_eval = pd.read_parquet('../datasets/processed/holdout_eval_v3d_vs_baselines.parquet')
    print(v3d_eval.to_string(index=False, float_format='%.4f'))
except FileNotFoundError:
    print('Pending: v3d retrain in progress. Re-run this cell after completion.')

Pending: v3d retrain in progress. Re-run this cell after completion.


## 9. Open / Pending

- [ ] v3d retrain + eval (in progress)
- [ ] 5-fold CV repeat on v3d
- [ ] Add v3d to OFF-truth eval
- [ ] Final cost reconcile via OpenRouter dashboard
- [ ] Update §6 of thesis with new numbers + lineage diagram
- [ ] Decide whether to keep gpt-5.5 Tier 2 in narrative (currently abandoned for v3d)

## Files / artifacts produced this session

```
src/experiments/
  build_off_derived_truth.py       — OFF nutriments → derived truth (56k rows)
  build_session_notebook.py        — this notebook builder
  cv_5fold_hybrid.py               — 5-fold CV with leak-free trainer
  eval_4way_v2_v3_v3b_v3c.py       — single-holdout 4-way comparator
  eval_holdout_v2_vs_v3.py         — v2-vs-v3 first comparator
  eval_off_truth_holdout.py        — OFF-truth 4-way evaluator
  extend_embeddings_b3.py          — B3 codes → silver + embeddings
  extend_embeddings_off_truth.py   — OFF-truth codes → silver + embeddings
  make_holdout_split.py            — stratified 80/20 split (+ tier column)
  merge_hybrid_v3.py               — v2 + Tier3 → consensus_hybrid_v3
  merge_v3d_corpus.py              — consensus_v3d builder (Opus promptfix + gemini promptfix)
  train_hybrid_cascade.py          — + tier-weighted, + leak-free (--holdout-codes)

models_backup/
  v2_clean/, v3_clean/, v3b_weighted/, v3c_no_silver/   — 44 .pkl each
  full_snapshot_2026-05-17/                              — 523 файла (229MB)

datasets/processed/
  consensus_holdout.parquet, consensus_v2_train.parquet, consensus_v3_train.parquet
  consensus_hybrid_v3.parquet, consensus_v3d.parquet
  holdout_eval_v2_vs_v3.parquet, holdout_eval_4way.parquet
  off_derived_truth.parquet, eval_off_truth_4way.parquet
  cv5fold_v3b.parquet
```
